    <a href="https://colab.research.google.com/github/alilouati/data-visualization-using-python/blob/main/Chapter_1/Ch1_Introduction_Foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

    # Chapter 1: Introduction & Foundations of Visual Encoding
    **Data Visualization — Ph.D. in Computing** · Dr. Ali Louati · IS Dept., PSAU

    ⏱️ Estimated lab time: **2 × 90 min**

    ### Learning objectives
    By the end of this lab you should be able to:
    1. Explain *why* summary statistics are not enough (Anscombe's quartet).
2. Choose a chart from the analytical question and the data type.
3. Rank visual channels by perceptual accuracy (Cleveland & McGill, 1984) and apply it.
4. Show how bin width / bandwidth choices change the story a distribution tells.
5. Handle over-plotting at scale (alpha, hexbin, 2-D density).
6. Choose perceptually uniform, colour-blind-safe colour maps and verify them numerically.
7. Detect and quantify misleading charts (Tufte's *lie factor*) and show uncertainty honestly.

    ### How to use this notebook
    * Run cells top to bottom with **Shift + Enter** (or *Runtime → Run all*).
    * Cells marked **✏️ Exercise** are for you. Cells marked **🔑 Solution** are collapsed: try first, then double-click to reveal.
    * Save your own copy: *File → Save a copy in Drive*.

## 0. Setup
Everything used here is pre-installed on Google Colab. The cell below only checks versions and sets a clean default style.

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns
IN_COLAB = "google.colab" in sys.modules
print(f"Colab: {IN_COLAB} | numpy {np.__version__} | pandas {pd.__version__} | matplotlib {mpl.__version__} | seaborn {sns.__version__}")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
rng = np.random.default_rng(42)   # fixed seed → reproducible figures

## 1. Why visualise? Anscombe's quartet
Four datasets with (almost) identical means, variances, correlation and regression line. Compute the statistics first, *then* look.

In [ ]:
ans = sns.load_dataset("anscombe")
summary = ans.groupby("dataset").agg(mean_x=("x", "mean"), mean_y=("y", "mean"),
                                     var_x=("x", "var"), var_y=("y", "var"))
summary["corr_xy"] = ans.groupby("dataset").apply(lambda d: d["x"].corr(d["y"]), include_groups=False)
summary.round(2)

In [ ]:
g = sns.lmplot(data=ans, x="x", y="y", col="dataset", col_wrap=4, ci=None, height=2.6,
               scatter_kws={"s": 35}, line_kws={"color": "C3"})
g.figure.suptitle("Same statistics, four different stories", y=1.04)
plt.show()

> **Discussion (PhD level).** Which of the four would break a linear model's assumptions, and how? Name a diagnostic plot that reveals each problem (residual plot, leverage plot, …). Matejka & Fitzmaurice (2017, *Same Stats, Different Graphs*) generalise this with simulated annealing — the "Datasaurus Dozen".

## 2. Start from the question, not the chart
| Analytical task | Data | Good first choice | Avoid |
|---|---|---|---|
| Compare categories | categorical × numeric | sorted bar / dot plot | 3-D bars, pies with many slices |
| Distribution | numeric | histogram + ECDF, box/violin per group | a single mean ± SD bar |
| Relationship | numeric × numeric | scatter (+ smoother), hexbin if *n* is large | line charts for unordered x |
| Change over time | time × numeric | line chart, small multiples | stacked areas for many series |
| Part-to-whole | categorical shares | stacked bar, waffle | pie with > 5 slices |
| Many variables | *p* ≫ 3 | small multiples, pair plot, PCA/UMAP map | 3-D scatter as the only view |

The helper below encodes this table so you can argue with it.

In [ ]:
def suggest_chart(task: str, n_points: int = 100, n_categories: int = 0) -> str:
    task = task.lower()
    if task == "distribution":
        return "ECDF + histogram" if n_points > 30 else "strip/swarm plot (show every point)"
    if task == "relationship":
        return "hexbin / 2-D density" if n_points > 5_000 else "scatter + LOWESS"
    if task == "comparison":
        return "dot plot (sorted)" if n_categories > 10 else "bar chart (sorted, zero baseline)"
    if task == "trend":
        return "small multiples of lines" if n_categories > 5 else "line chart"
    if task == "composition":
        return "stacked bar" if n_categories > 4 else "stacked bar or (sparingly) donut"
    return "start by writing the question in one sentence"

for t in ["distribution", "relationship", "comparison", "trend", "composition"]:
    print(f"{t:<13} → {suggest_chart(t, n_points=20_000, n_categories=6)}")

## 3. Perception: not all visual channels are equal
Cleveland & McGill (1984) measured how accurately people decode quantities. Approximate ranking (most → least accurate):

**position on common scale › position on unaligned scales › length › angle/slope › area › volume › colour saturation/hue**

Below, the **same five numbers** are encoded three ways. Try to rank B, C, D before reading the values.

In [ ]:
vals = pd.Series([23, 21, 19, 18, 19], index=list("ABCDE"))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].bar(vals.index, vals.values, color="C0"); axes[0].set_title("Position / length")
axes[1].pie(vals.values, labels=vals.index, startangle=90, counterclock=False,
            wedgeprops=dict(edgecolor="white")); axes[1].set_title("Angle / area")
axes[2].scatter(range(5), [0]*5, s=vals.values**2 * 3, alpha=.6)
for i, k in enumerate(vals.index): axes[2].text(i, 0, k, ha="center", va="center")
axes[2].set_xlim(-0.7, 4.7); axes[2].axis("off"); axes[2].set_title("Area (bubbles)")
plt.tight_layout(); plt.show()

**Stevens' power law.** Perceived magnitude ≈ *k·stimulus*<sup>β</sup>. For length β ≈ 1; for area β ≈ 0.7; for volume β ≈ 0.6. So a circle with 2× the area *looks* only ≈ 1.6× larger.

In [ ]:
ratio = np.linspace(1, 10, 100)
for name, beta in [("length (β≈1.0)", 1.0), ("area (β≈0.7)", 0.7), ("volume (β≈0.6)", 0.6)]:
    plt.plot(ratio, ratio**beta, label=name)
plt.plot(ratio, ratio, "k:", lw=1)
plt.xlabel("true ratio"); plt.ylabel("perceived ratio"); plt.title("Stevens' power law"); plt.legend(); plt.show()

### ✏️ Exercise 1 — a mini perception experiment
1. Generate 20 random pairs of values. For each pair, show them as **two bars** and as **two circles** (area ∝ value).
2. Record your own estimate of the ratio for each (write them in a list).
3. Fit β by regressing log(perceived) on log(true) for each channel. Do you reproduce Stevens' exponents?
4. *Research angle:* how would you design this as a proper crowdsourced study (Heer & Bostock, 2010)? What are the threats to validity?

In [ ]:
# ✏️ Your code here

In [ ]:
#@title 🔑 Solution sketch — fitting β (try first, then double-click to reveal) { display-mode: "form" }
true = rng.uniform(1.2, 5, 20)
perceived_area = true**0.7 * rng.lognormal(0, 0.08, 20)   # replace with YOUR estimates
beta, logk = np.polyfit(np.log(true), np.log(perceived_area), 1)
print(f"estimated β = {beta:.2f}")

## 4. Distributions: your parameter choices are part of the argument
### 4.1 Histogram bin width
The same bimodal sample looks unimodal with bins that are too wide. Compare rules of thumb.

In [ ]:
x = np.concatenate([rng.normal(0, 1, 400), rng.normal(3, 0.6, 200)])
rules = ["sturges", "fd", "scott", 5, 80]
fig, axes = plt.subplots(1, len(rules), figsize=(15, 2.8), sharey=False)
for ax, r in zip(axes, rules):
    edges = np.histogram_bin_edges(x, bins=r)
    ax.hist(x, bins=edges, color="C0", edgecolor="white")
    ax.set_title(f"bins='{r}' → {len(edges)-1}")
plt.tight_layout(); plt.show()

### 4.2 KDE bandwidth and the ECDF (no tuning parameter at all)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for bw in [0.2, 0.5, 1, 2]:
    sns.kdeplot(x, bw_adjust=bw, ax=axes[0], label=f"bw_adjust={bw}")
axes[0].legend(); axes[0].set_title("KDE: bandwidth changes the shape")
sns.ecdfplot(x, ax=axes[1]); axes[1].set_title("ECDF: every point, no smoothing")
plt.tight_layout(); plt.show()

> **Takeaway.** Report the bin rule / bandwidth in your figure caption. For rigorous comparison of distributions, pair the histogram with an ECDF.

## 5. Relationships at scale: over-plotting
With 100 000 points a scatter plot becomes an ink blob. Compare four remedies.

In [ ]:
n = 100_000
a = rng.normal(size=n); b = 0.6 * a + rng.normal(scale=0.8, size=n)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharex=True, sharey=True)
axes[0].scatter(a, b, s=2); axes[0].set_title("raw scatter")
axes[1].scatter(a, b, s=2, alpha=0.03); axes[1].set_title("alpha = 0.03")
hb = axes[2].hexbin(a, b, gridsize=50, bins="log", cmap="viridis"); axes[2].set_title("hexbin (log counts)")
sns.kdeplot(x=a[:5000], y=b[:5000], fill=True, levels=10, cmap="mako", ax=axes[3]); axes[3].set_title("2-D KDE (5k sample)")
fig.colorbar(hb, ax=axes[2], label="log10(count)")
plt.tight_layout(); plt.show()

## 6. Colour with a purpose
* **Qualitative** palettes for categories (≤ 8 distinguishable hues).
* **Sequential** (light → dark) for magnitudes.
* **Diverging** for deviations from a meaningful centre (0, mean, target).

### 6.1 Why `jet`/rainbow is harmful: check perceptual lightness
We convert each colour map to CIE L\* (perceived lightness). A good sequential map is **monotonic** in L\*.

In [ ]:
def srgb_to_lab_L(rgb):
    """CIE L* from sRGB in [0,1] (D65)."""
    rgb = np.asarray(rgb)[..., :3]
    lin = np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    Y = lin @ np.array([0.2126, 0.7152, 0.0722])
    f = np.where(Y > (6/29)**3, np.cbrt(Y), Y / (3 * (6/29)**2) + 4/29)
    return 116 * f - 16

t = np.linspace(0, 1, 256)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for name in ["jet", "viridis", "cividis", "magma", "turbo"]:
    axes[0].plot(t, srgb_to_lab_L(mpl.colormaps[name](t)), label=name)
axes[0].set_xlabel("position in colour map"); axes[0].set_ylabel("L* (lightness)"); axes[0].legend()
axes[0].set_title("Perceptual lightness profile")

xx, yy = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(-3, 3, 300))
zz = np.exp(-(xx**2 + yy**2)) + 0.4 * np.exp(-((xx - 1.5)**2 + (yy - 1)**2) * 4)
axes[1].axis("off"); axes[1].set_title("Same smooth field: jet (left) vs viridis (right)")
for k, cm in enumerate(["jet", "viridis"]):
    sub = axes[1].inset_axes([k * 0.5, 0, 0.48, 1]); sub.imshow(zz, cmap=cm); sub.axis("off")
plt.tight_layout(); plt.show()
print("jet shows bright yellow/cyan bands = false 'edges'; viridis changes lightness smoothly.")

### 6.2 Simulating colour-vision deficiency (CVD)
≈ 8 % of men have a red–green deficiency. We simulate deuteranopia and protanopia with the matrices of Machado, Oliveira & Fernandes (2009), applied in linear RGB.

In [ ]:
CVD = {
    "deuteranopia": np.array([[0.367322, 0.860646, -0.227968], [0.280085, 0.672501, 0.047413], [-0.011820, 0.042940, 0.968881]]),
    "protanopia":   np.array([[0.152286, 1.052583, -0.204868], [0.114503, 0.786281, 0.099216], [-0.003882, -0.048116, 1.051998]]),
}
def simulate_cvd(rgb, kind):
    rgb = np.asarray(rgb)[..., :3]
    lin = np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    out = np.clip(lin @ CVD[kind].T, 0, 1)
    return np.where(out <= 0.0031308, 12.92 * out, 1.055 * out ** (1 / 2.4) - 0.055)

palettes = {"red/green (bad)": ["#d62728", "#2ca02c", "#ff7f0e", "#8c564b"],
            "Okabe–Ito (safe)": ["#E69F00", "#56B4E9", "#009E73", "#CC79A7"]}
fig, axes = plt.subplots(2, 3, figsize=(10, 2.6))
for r, (pname, hexes) in enumerate(palettes.items()):
    rgb = np.array([mpl.colors.to_rgb(h) for h in hexes])[None]
    for c, (title, img) in enumerate([("normal", rgb), ("deuteranopia", simulate_cvd(rgb, "deuteranopia")),
                                      ("protanopia", simulate_cvd(rgb, "protanopia"))]):
        axes[r, c].imshow(img, aspect="auto"); axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0: axes[r, c].set_title(title)
    axes[r, 0].set_ylabel(pname, rotation=0, ha="right", va="center")
plt.tight_layout(); plt.show()

### ✏️ Exercise 2
Write `min_pairwise_distance(hexes, kind)` that returns the smallest Euclidean distance between any two colours of a palette **after** CVD simulation (use L\*a\*b\* for bonus marks). Use it to rank `tab10`, `Set2`, `colorblind` (seaborn) and Okabe–Ito.

In [ ]:
# ✏️ Your code here

In [ ]:
#@title 🔑 Solution (try first, then double-click to reveal) { display-mode: "form" }
from itertools import combinations
def min_pairwise_distance(hexes, kind=None):
    rgb = np.array([mpl.colors.to_rgb(h) for h in hexes])
    if kind: rgb = simulate_cvd(rgb, kind)
    return min(np.linalg.norm(p - q) for p, q in combinations(rgb, 2))

cands = {"tab10[:6]": [mpl.colors.to_hex(c) for c in mpl.colormaps["tab10"].colors[:6]],
         "Set2[:6]": [mpl.colors.to_hex(c) for c in mpl.colormaps["Set2"].colors[:6]],
         "colorblind[:6]": sns.color_palette("colorblind", 6).as_hex(),
         "Okabe-Ito": ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00"]}
pd.DataFrame({k: {kind: round(min_pairwise_distance(v, kind if kind != "normal" else None), 3)
                  for kind in ["normal", "deuteranopia", "protanopia"]} for k, v in cands.items()}).T

## 7. Honesty: truncated axes and the lie factor
Tufte's **lie factor** = (size of effect shown in graphic) / (size of effect in data). Values far from 1 indicate distortion.

In [ ]:
cats, v = ["2022", "2023"], [50, 54]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].bar(cats, v, color=["#bbb", "C3"]); axes[0].set_ylim(48, 55); axes[0].set_title("Truncated (y from 48)")
axes[1].bar(cats, v, color=["#bbb", "C3"]); axes[1].set_ylim(0, 60); axes[1].set_title("Zero baseline")
plt.tight_layout(); plt.show()

def lie_factor(v0, v1, axis_min):
    data_effect = (v1 - v0) / v0
    graphic_effect = ((v1 - axis_min) - (v0 - axis_min)) / (v0 - axis_min)
    return graphic_effect / data_effect
print(f"lie factor (truncated): {lie_factor(50, 54, 48):.1f}   |   lie factor (zero): {lie_factor(50, 54, 0):.1f}")

> Bars encode quantity by **length**, so they need a zero baseline. Line charts encode **change** by position/slope, so a non-zero baseline is often fine — but label it.

## 8. Showing uncertainty
Error bars are ambiguous unless you say what they are. Compare SD, standard error and a **bootstrap 95 % CI** on the `tips` data.

In [ ]:
tips = sns.load_dataset("tips")
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4), sharey=True)
for ax, (eb, title) in zip(axes, [("sd", "± 1 SD (spread)"), ("se", "± 1 SE"), (("ci", 95), "95 % bootstrap CI"), (("pi", 95), "95 % percentile interval")]):
    sns.pointplot(data=tips, x="day", y="tip", errorbar=eb, ax=ax, capsize=.2, seed=0)
    ax.set_title(title)
plt.tight_layout(); plt.show()

In [ ]:
# Raw data + uncertainty together is usually the most honest view
ax = sns.stripplot(data=tips, x="day", y="tip", alpha=.35, jitter=.25, color="grey")
sns.pointplot(data=tips, x="day", y="tip", errorbar=("ci", 95), color="C3", ax=ax, linestyle="none", capsize=.15, seed=0)
ax.set_title("Every observation + mean with 95 % CI"); plt.show()

## 9. From plots to dashboards (preview)
A single plot answers **one** question. A dashboard combines KPIs + coordinated views + filters (Shneiderman's mantra: *overview first, zoom and filter, details on demand*). Chapters 4–5 build these in Dash, Bokeh and Streamlit — the GRAALI case study (Graduation-project management at CCES, PSAU) is our running real-world example.

## 10. ✏️ Take-home assignment
Pick a public dataset relevant to **your own PhD topic**.
1. Write three analytical questions (one each: distribution, relationship, comparison).
2. Produce one figure per question. Justify every encoding choice with a reference to perception research.
3. Produce one deliberately **misleading** version of one figure and compute its lie factor.
4. Run the CVD check from Exercise 2 on your palette.
5. Submit the notebook (*File → Download → .ipynb*) with a 200-word critique.

## Quick check
1. Which channel is decoded most accurately: position, angle or area?
2. Why must a bar chart start at zero but a line chart need not?
3. What does an ECDF show that a histogram hides?
4. Name two remedies for over-plotting and one cost of each.
5. Why is `jet` a poor sequential colour map? Give a measurable criterion.

## References
* Cleveland, W. & McGill, R. (1984). *Graphical Perception*. JASA 79(387).
* Heer, J. & Bostock, M. (2010). *Crowdsourcing Graphical Perception*. CHI.
* Machado, G., Oliveira, M., Fernandes, L. (2009). *A Physiologically-based Model for Simulation of Color Vision Deficiency*. IEEE TVCG.
* Matejka, J. & Fitzmaurice, G. (2017). *Same Stats, Different Graphs*. CHI.
* Munzner, T. (2014). *Visualization Analysis and Design*. CRC Press.
* Tufte, E. (2001). *The Visual Display of Quantitative Information*, 2nd ed.